In [1]:
%cd /drive2/ryusejong/LFF
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
import json 
import time 
import re
import random
import numpy as np 
from tqdm.auto import tqdm
from util.utils import set_seed, read_data, save_result, get_answer_from_text, chat_huggingface, construct_conversation
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel

seed = 42
set_seed(seed)

/drive2/ryusejong/LFF


/drive2/ryusejong/miniconda3/envs/llm1/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [82]:
# Base & Output file
base_path = "output/zeroshot_CoT/GSM8K_Llama-3-8B-Instruct_zeroshot_CoT_test_512_seed42_portion0.1.jsonl"
base_file = read_data(base_path)

new_path = "output/LFF_v7/GSM8K_Llama-3-8B-Instruct_LFF_v7_10_test_512_seed42_portion0.1.jsonl"
new_file = read_data(new_path)

print(f"length of base_file: {len(base_file)}")
print(f"length of output_file: {len(new_file)}")

length of base_file: 131
length of output_file: 131


In [83]:
# Base(zeroshot_CoT): number of failure samples
corrects = []
incorrects = []

for i in tqdm(range(len(base_file))):
    true_answer = base_file[i]["answer"]
    base_pred_answer = base_file[i]["pred_ans"]

    if base_pred_answer == true_answer:
        corrects.append(base_file[i])
    else:
        incorrects.append(base_file[i])
            
print(f"correct: {len(corrects)}\nIndex: {[o['index'] for o in corrects]}\n")
print(f"incorrect: {len(incorrects)}\nIndex: {[o['index'] for o in incorrects]}\n")
print(f"total num: {len(corrects) + len(incorrects)}")

100%|██████████| 131/131 [00:00<00:00, 440267.49it/s]

correct: 105
Index: [0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 15, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27, 29, 30, 31, 32, 34, 35, 36, 37, 39, 42, 43, 45, 46, 47, 48, 49, 55, 56, 57, 60, 61, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 94, 96, 97, 98, 99, 100, 102, 103, 104, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129]

incorrect: 26
Index: [7, 14, 16, 20, 28, 33, 38, 40, 41, 44, 50, 51, 52, 53, 54, 58, 59, 62, 63, 81, 92, 93, 95, 101, 105, 130]

total num: 131


In [84]:
# Ouptut: number of retrieved samples
retrieves = []
not_retrieves = []

for i in tqdm(range(len(new_file))):

    if "pred_ans2" in new_file[i]:
        retrieves.append(new_file[i])
    else:
        not_retrieves.append(new_file[i])
            
print(f"retrieve: {len(retrieves)}\nIndex: {[o['index'] for o in retrieves]}\n")
print(f"not retrieve: {len(not_retrieves)}\nIndex: {[o['index'] for o in not_retrieves]}\n")
print(f"total num: {len(retrieves) + len(not_retrieves)}")

100%|██████████| 131/131 [00:00<00:00, 356094.51it/s]

retrieve: 37
Index: [2, 3, 5, 8, 13, 16, 17, 25, 31, 33, 35, 36, 40, 44, 49, 52, 54, 55, 56, 57, 58, 65, 70, 78, 80, 82, 85, 86, 88, 92, 93, 98, 100, 109, 120, 125, 129]

not retrieve: 94
Index: [0, 1, 4, 6, 7, 9, 10, 11, 12, 14, 15, 18, 19, 20, 21, 22, 23, 24, 26, 27, 28, 29, 30, 32, 34, 37, 38, 39, 41, 42, 43, 45, 46, 47, 48, 50, 51, 53, 59, 60, 61, 62, 63, 64, 66, 67, 68, 69, 71, 72, 73, 74, 75, 76, 77, 79, 81, 83, 84, 87, 89, 90, 91, 94, 95, 96, 97, 99, 101, 102, 103, 104, 105, 106, 107, 108, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 121, 122, 123, 124, 126, 127, 128, 130]

total num: 131


In [85]:
# number of failure samples in retrieved samples
retrieves_corrects = []
retrieves_incorrects = []

for i in tqdm(range(len(retrieves))):
    true_answer = retrieves[i]["answer"]
    base_pred_answer = retrieves[i]["pred_ans1"]

    if base_pred_answer == true_answer:
        retrieves_corrects.append(retrieves[i])
    else:
        retrieves_incorrects.append(retrieves[i])

print(f"retrieve_correct: {len(retrieves_corrects)}\nIndex: {[o['index'] for o in retrieves_corrects]}\n")
print(f"retrieve_incorrect: {len(retrieves_incorrects)}\nIndex: {[o['index'] for o in retrieves_incorrects]}\n")
print(f"total num: {len(retrieves_corrects) + len(retrieves_incorrects)}\n")
print(f"retrieve ratio: {(len(retrieves_incorrects) / len(retrieves))*100:.2f}%")

100%|██████████| 37/37 [00:00<00:00, 213759.29it/s]

retrieve_correct: 28
Index: [2, 3, 5, 8, 13, 17, 25, 31, 35, 36, 49, 55, 56, 57, 65, 70, 78, 80, 82, 85, 86, 88, 98, 100, 109, 120, 125, 129]

retrieve_incorrect: 9
Index: [16, 33, 40, 44, 52, 54, 58, 92, 93]

total num: 37

retrieve ratio: 24.32%


In [86]:
# correction rate of retrieve_incorrects
correct_successes = []

for i in tqdm(range(len(retrieves_incorrects))):
    true_answer = retrieves_incorrects[i]["answer"]
    new_pred_answer = retrieves_incorrects[i]["pred_ans2"]

    if new_pred_answer == true_answer:
        correct_successes.append(retrieves_incorrects[i])
        
print(f"correct_success: {len(correct_successes)}\nIndex: {[o['index'] for o in correct_successes]}\n")
print(f"correction ratio: {(len(correct_successes) / len(retrieves_incorrects))*100:.2f}%\n")

# incorrection rate of retrieve_corrects
incorrect_failures = []

for i in tqdm(range(len(retrieves_corrects))):
    true_answer = retrieves_corrects[i]["answer"]
    new_pred_answer = retrieves_corrects[i]["pred_ans2"]

    if new_pred_answer != true_answer:
        incorrect_failures.append(retrieves_corrects[i])
        
print(f"incorrect_failure: {len(incorrect_failures)}\nIndex: {[o['index'] for o in incorrect_failures]}\n")
print(f"incorrection ratio: {(len(incorrect_failures) / len(retrieves_corrects))*100:.2f}%\n")

100%|██████████| 9/9 [00:00<00:00, 77353.97it/s]


correct_success: 2
Index: [44, 93]

correction ratio: 22.22%



100%|██████████| 28/28 [00:00<00:00, 271225.20it/s]

incorrect_failure: 9
Index: [13, 25, 31, 78, 82, 85, 120, 125, 129]

incorrection ratio: 32.14%

